# AgentOps: observe, evaluate, improve

The application is **Northstar Support**. This notebook is its engineering workspace.

- **LangGraph** executes the agent and pauses sensitive actions for approval.
- **LangSmith** records the run: model calls, tools, timing, tokens and evaluation feedback.
- **DeepEval** scores results. ExactMatch checks the local baseline; G-Eval uses an LLM judge for answer quality.

Select **SupportOps (.venv)**. Run cells in order. Live calls are off by default.

```text
Question -> agent -> tools -> answer
              |                 |
              +-- LangSmith ----+   What happened?
                                |
                           DeepEval  Was it correct?
                                |
                        report -> improve -> compare
```

## 1. Load the engineering tools

This cell loads configuration without printing keys. A configured key does not prove the server is reachable. An actual experiment verifies connectivity.

In [ ]:
import html
import json
from pathlib import Path
from IPython.display import HTML, display
from support_ops.config import find_project_root
from support_ops.observability import tracing_status, version_labels
from support_ops.agent_evaluation import load_agent_scenarios, run_suite, report_passed

root = find_project_root()
cases = load_agent_scenarios(root)
print(tracing_status(root))
print(version_labels(root))

def table(rows):
    if not rows:
        print("No results yet.")
        return
    keys = list(rows[0])
    header = "".join(f"<th>{html.escape(str(k))}</th>" for k in keys)
    body = "".join("<tr>" + "".join(f"<td>{html.escape(str(row.get(k, '')))}</td>" for k in keys) + "</tr>" for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))

## 2. See the measurement flow

The customer interface stays focused on support. Traces and evaluation results belong here and in LangSmith. The judge uses a separate model call; its tokens are not included in agent tokens.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.set(xlim=(0, 11), ylim=(0, 4))
ax.axis("off")
boxes = [(0.2, 2.5, "Versioned questions"), (3.9, 2.5, "LangGraph agent"), (7.6, 2.5, "Answer + tool results"),
         (0.2, 0.5, "Local JSON report"), (3.9, 0.5, "DeepEval scores"), (7.6, 0.5, "LangSmith experiment")]
for x, y, label in boxes:
    ax.add_patch(FancyBboxPatch((x,y), 3, .8, boxstyle="round,pad=0.1", facecolor="#10233d", edgecolor="#10233d"))
    ax.text(x+1.5,y+.4,label,ha="center",va="center",color="white",fontsize=10)
for a,b in [((3.3,2.9),(3.8,2.9)),((7,2.9),(7.5,2.9)),((9.1,2.4),(9.1,1.4)),((7.5,.9),(7,.9)),((3.8,.9),(3.3,.9))]:
    ax.annotate("",xy=b,xytext=a,arrowprops={"arrowstyle":"->","color":"#168f83","lw":2})
plt.tight_layout()
plt.show()

## 3. Inspect the test set

Each case contains a question, the expected tool and verified result fields, an answer reference, and optional approval. Every live case uses a new temporary database. Evaluations do not cancel orders in the application.

The access override case accepts either a safe refusal or an access-checked lookup. A refusal does not need a tool call.

In [ ]:
table([{"case": c.id, "tool": c.required_tool or "none", "approval": c.approval, "question": c.question} for c in cases])

## 4. Run the local baseline

No LLM is called. Ten reviewed questions check the earlier deterministic graph. DeepEval ExactMatch scores the answer, while Python checks the route. These results do not measure the real LLM's quality.

In [ ]:
offline_report, offline_path = run_suite(root)
table(offline_report["offline"]["cases"])
assert report_passed(offline_report)
print("Saved:", offline_path.relative_to(root))

## 5. Understand the live checks

| Check | What it verifies |
| --- | --- |
| tool_selection | Expected tool, or no tool when clarification is needed |
| tool_result | Retrieved facts match reviewed fields |
| approval | Required pause happened before any update |
| state_integrity | Only the expected order status changed |
| no_foreign_profile | Another customer's name, email and address are absent |
| tool_budget | The agent stays within its allowed number of calls |
| answer_present | A final response exists |
| DeepEval G-Eval | The response answers the question using the reference and tool evidence |

A score of **0.7 or higher** passes the judge threshold. A judge score cannot override a failed access or approval check. The profile check covers specified fixture values; it is not a general proof against every possible data leak.

`passed` means all enabled checks passed. `failed` means behavior or answer quality failed. `error` means execution or judging could not finish, for example because of an API quota. Missing scores are never treated as success.

## 6. Run a real experiment (optional)

Start with one case. Set `RUN_LIVE = True` to use Groq, `USE_JUDGE = True` to enable DeepEval G-Eval, and `PUBLISH_EXPERIMENT = True` to write the synthetic dataset and results to your private LangSmith workspace.

Set `CASE_IDS = None` to run all nine cases. Calls are sequential and spaced to reduce quota errors. Every completed case is saved immediately, even if a later case fails. There are no automatic retries of actions.

```text
one case -> fresh database -> real agent -> objective checks -> optional judge
                                                            -> report + experiment
```

In [ ]:
RUN_LIVE = False
USE_JUDGE = False
PUBLISH_EXPERIMENT = False
CASE_IDS = ["order_status"]

if RUN_LIVE:
    live_report, live_path = run_suite(
        root, live=True, judge_enabled=USE_JUDGE, publish=PUBLISH_EXPERIMENT,
        case_ids=CASE_IDS, delay_seconds=20,
    )
    print("Passed:", report_passed(live_report))
    print("Saved:", live_path.relative_to(root))
    if live_report.get("experiment_url"):
        display(HTML(f'<a href="{html.escape(live_report["experiment_url"])}" target="_blank">Open your private LangSmith experiment</a>'))
else:
    print("Live calls disabled. Continue to inspect an existing report.")

## 7. Read the latest completed live report

This cell reads a local file only. It does not call Groq or LangSmith. Agent time measures the agent and its tools; judge time and deliberate quota delays are separate. Tokens count only successful responses with provider usage metadata. Failed calls may have consumed tokens that were not reported.

In [ ]:
saved = []
for path in (root / "artifacts/evaluations").glob("*.json"):
    value = json.loads(path.read_text(encoding="utf-8"))
    if value.get("mode") == "live" and value.get("status") != "running":
        saved.append((path, value))
saved.sort(key=lambda entry: entry[1]["created_at"])
latest = saved[-1][1] if saved else None
if latest:
    print("Report time:", latest["created_at"], "Model:", latest["model"])
    table([{"case": r["id"], "status": r["status"], "agent_seconds": r["agent_seconds"],
            "agent_tokens": (r.get("usage") or {}).get("total_tokens"),
            "judge_score": r.get("judge", {}).get("score"),
            "judge_tokens": r.get("judge", {}).get("total_tokens"),
            "error": r.get("error_type") or r.get("judge", {}).get("error_type", "")}
           for r in latest["cases"]])
else:
    print("No live report yet. Run section 6 or the CLI command below.")

## 8. Inspect failures and performance

Fix the cause before rerunning. A failed objective check points to the tool or workflow. A failed judge score needs human review of the answer, reference and judge reason. A quota error needs a later retry, not a changed expected answer.

In [ ]:
if latest and latest["cases"]:
    for row in latest["cases"]:
        if row["status"] != "passed":
            print(row["id"], row["status"], {k:v for k,v in row.get("checks", {}).items() if not v})
            print(row.get("judge", {}).get("reason") or row.get("error_type") or row.get("judge", {}).get("error_type"))
    labels = [r["id"] for r in latest["cases"]]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(labels, [r["agent_seconds"] for r in latest["cases"]], color="#168f83")
    ax.set_xlabel("Agent seconds (judge and quota delays excluded)")
    ax.set_title("Measured live execution time")
    plt.tight_layout()
    plt.show()
else:
    print("Run a live evaluation to see measured performance.")

## 9. Read LangSmith

1. Open your LangSmith workspace and the **SupportOps** tracing project.
2. Send a question from the app. Open its `support_request` trace.
3. Expand `agent`, the Groq model call, `execute_tools`, and the selected tool. Inspect arguments, returned facts, tokens and timing.
4. For cancellation, inspect the paused run and resumed run. The same conversation ID groups them; approval is not a second free-form LLM decision.
5. Under datasets/experiments, open the `SupportOps-agent-...` dataset. The latest experiment has the objective checks, `case_passed`, and `deepeval_grounding` when judging is enabled.
6. Open a failed row and compare the question, reference, actual answer and feedback.

Normal app traces and evaluation experiments have separate project names. The metadata stores model, prompt, data/configuration and implementation fingerprints. No API key is included.

[LangSmith evaluation API](https://reference.langchain.com/python/langsmith/client/Client/evaluate) | [DeepEval G-Eval](https://deepeval.com/docs/metrics-llm-evals)

## 10. Compare changes honestly

Run the same selected cases before and after a prompt or implementation change. Use LangSmith's comparison view, or inspect both saved reports below. Keep questions, references, fixtures, judge and threshold fixed when comparing prompt variants. Different suite sizes are not directly comparable. One successful run is not a reliability estimate.

The judge currently uses the same model family as the agent. This keeps setup simple, but their errors may be correlated. Keep human-reviewed references and inspect samples manually; stronger evidence requires repeated runs and an independent judge.

In [ ]:
table([{"time": r["created_at"], "cases": len(r["cases"]),
        "passed": sum(c["status"] == "passed" for c in r["cases"]),
        "prompt": r["prompt_version"], "data_and_config": r["evaluation_version"],
        "implementation": r.get("implementation_version", "unknown")}
       for _, r in saved[-5:]])

## Terminal commands

Run from the repository root in PowerShell:

```powershell
.\.venv\Scripts\python.exe -m pytest -q
.\.venv\Scripts\python.exe scripts/run_evals.py
.\.venv\Scripts\python.exe scripts/run_evals.py --live --judge --langsmith --case order_status --delay 20
.\.venv\Scripts\python.exe scripts/run_evals.py --live --judge --langsmith --delay 20
```

Reports are stored in `artifacts/evaluations/` and stay outside Git. The CLI exits with code 1 when a required check fails or an execution is incomplete. Share reviewed aggregate results; never publish keys, raw environment files or private account links.